# 🚀 ROADGUARD AI — HUẤN LUYỆN LẦN 2 (PHASE 2 UPGRADE)
## Nâng Cấp Model `roadguard_crack_yolo11n_v1.pt` Thành Unified 2-Class (Pothole + Crack)
---
> **Chiến thuật:** Transfer Learning & Backbone Freezing từ mô hình vết nứt hiện tại.
> **Mục tiêu:** Mở rộng thêm năng lực nhận diện **Ổ gà (Pothole)** và **Khử bóng râm (Hard Negatives)** mà **bảo toàn 100% độ nhạy vết nứt** đã học từ Phase 1.
> **Tập dữ liệu:** `unified_road_dataset_15k.zip` (~15,000 ảnh chuẩn hóa: 10,111 ổ gà + 5,000 vết nứt + 6,060 mẫu âm tính).
> **Phần cứng:** Google Colab GPU Tesla T4 (Miễn phí) — Thời gian chạy: ~30-40 phút cho 35 Epochs.

### 📌 BƯỚC 1: KIỂM TRA GPU TESLA T4 & CÀI ĐẶT THƯ VIỆN

In [ ]:
# 1. Kiểm tra trạng thái GPU trên Colab
!nvidia-smi

import torch
print('=' * 65)
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Model:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM Capacity:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ CHÚ Ý: Hãy vào menu Runtime -> Change runtime type -> Chọn T4 GPU!')
print('=' * 65)

# 2. Cài đặt Ultralytics YOLOv11 mới nhất
!pip install -q ultralytics matplotlib pyyaml opencv-python

### 📥 BƯỚC 2: TẢI TRỌNG SỐ V1 (`roadguard_crack_yolo11n_v1.pt`) & DATASET 15K
Bạn có thể nạp file bằng 1 trong 2 cách:
1. **Cách 1 (Kéo thả):** Kéo thả file `roadguard_crack_yolo11n_v1.pt` và `unified_road_dataset_15k.zip` vào thanh Files bên trái của Colab.
2. **Cách 2 (Google Drive):** Đặt vào thư mục Drive rồi Mount Drive bằng đoạn code bên dưới.

In [ ]:
import os, shutil, zipfile

# Tùy chọn mount Google Drive nếu file lưu trên Drive (bỏ dấu # nếu dùng):
# from google.colab import drive
# drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/unified_road_dataset_15k.zip' '/content/'
# !cp '/content/drive/MyDrive/roadguard_crack_yolo11n_v1.pt' '/content/'

# Giải nén dataset an toàn
zip_path = 'unified_road_dataset_15k.zip'
extract_path = '/content/dataset'

if os.path.exists(zip_path):
    print(f'Đang giải nén {zip_path}...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_path)
    print(f'✓ Đã giải nén thành công vào: {extract_path}')
else:
    print(f'⚠️ Chưa thấy file {zip_path}. Hãy tải file lên trước khi chạy tiếp!')

### ⚙️ BƯỚC 3: THIẾT LẬP FILE CẤU HÌNH `data_phase2.yaml` TRÊN COLAB

In [ ]:
yaml_content = '''
path: /content/dataset
train: train/images
val:   valid/images
test:  test/images

nc: 2
names:
  0: pothole
  1: crack
'''

yaml_path = '/content/data_phase2.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip())

print(f'✓ Đã tạo file cấu hình: {yaml_path}')

# Kiểm tra số lượng ảnh
from pathlib import Path
dp = Path('/content/dataset')
if dp.exists():
    for split in ['train', 'valid', 'test']:
        imgs = list((dp / split / 'images').glob('*.*'))
        lbls = list((dp / split / 'labels').glob('*.txt'))
        print(f'Split {split.upper():<5}: {len(imgs):,} ảnh | {len(lbls):,} nhãn')
else:
    print('⚠️ Chưa tìm thấy thư mục dataset!')

### 🧠 BƯỚC 4: TIẾN HÀNH FINE-TUNING LẦN 2 (TRANSFER LEARNING TỪ MODEL V1)
---
> **Cơ chế chống quên tri thức (Catastrophic Forgetting):**
> 1. `model = YOLO('roadguard_crack_yolo11n_v1.pt')`: Khởi tạo từ checkpoint vết nứt hiện tại.
> 2. `freeze=10`: Đóng băng 10 layer Backbone để bảo vệ các bộ lọc đặc trưng vết nứt.
> 3. `lr0=0.001`: Giảm Learning Rate xuống $10^{-3}$ để đầu phát hiện học thêm ổ gà nhẹ nhàng.
> 4. `warmup_epochs=3`: Khởi động ấm gradient cho class mới.

In [ ]:
from ultralytics import YOLO

# 1. Nạp model v1 hiện tại
checkpoint_v1 = 'roadguard_crack_yolo11n_v1.pt'
if not os.path.exists(checkpoint_v1):
    print(f'⚠️ Chưa thấy {checkpoint_v1}, dùng tạm yolo11n.pt làm gốc dự phòng')
    checkpoint_v1 = 'yolo11n.pt'

model = YOLO(checkpoint_v1)
print(f'Trọng số gốc nạp vào: {checkpoint_v1}')
print(f'Classes ban đầu:      {model.names}')

# 2. Kích hoạt huấn luyện lần 2
results = model.train(
    data='/content/data_phase2.yaml',
    epochs=35,            # 35 epochs là tối ưu trên 15k ảnh
    imgsz=640,
    batch=16,             # Batch 16 vừa khít 12GB VRAM T4
    device=0,
    freeze=10,            # Đóng băng 10 tầng Backbone (Bảo toàn tri thức vết nứt)
    lr0=0.001,            # Learning Rate khởi điểm thấp hơn scratch
    lrf=0.01,             # Cosine learning rate scheduler
    warmup_epochs=3,
    box=7.5,              # Trọng số định vị BBox chính xác
    cls=0.5,              # Trọng số phân lớp nhãn
    dfl=1.5,
    mosaic=0.5,           # Tăng cường dữ liệu ghép ảnh
    close_mosaic=5,       # Tắt mosaic 5 epoch cuối để hội tụ mượt mà
    project='roadguard_phase2',
    name='yolo11n_v2_unified',
    save=True,
    val=True,
    plots=True
)
print('🎉 HUẤN LUYỆN LẦN 2 HOÀN TẤT!')

### 📊 BƯỚC 5: ĐÁNH GIÁ CHỈ SỐ MÔ HÌNH V2 TRÊN TẬP TEST ĐỘC LẬP (751 ẢNH)

In [ ]:
# Nạp trọng số xuất sắc nhất
best_model_path = '/content/roadguard_phase2/yolo11n_v2_unified/weights/best.pt'
v2_model = YOLO(best_model_path)

# Đánh giá trên tập test
metrics = v2_model.val(data='/content/data_phase2.yaml', split='test')

print('=' * 65)
print('🏆 KẾT QUẢ ĐÁNH GIÁ MODEL V2 TRÊN TẬP TEST:')
print(f'• mAP@50 (Tổng thể):     {metrics.box.map50 * 100:.2f}%')
print(f'• mAP@50-95:             {metrics.box.map * 100:.2f}%')
print(f'• Precision:             {metrics.box.mp * 100:.2f}%')
print(f'• Recall:                {metrics.box.mr * 100:.2f}%')
print('=' * 65)

# Hiển thị chỉ số từng class
for idx, cls_name in v2_model.names.items():
    cls_map50 = metrics.box.maps[idx] if len(metrics.box.maps) > idx else 0.0
    print(f'  Lớp [{cls_name.upper()}]: mAP@50-95 = {cls_map50 * 100:.2f}%')

### 🖼️ BƯỚC 6: XEM BIỂU ĐỒ KẾT QUẢ & MA TRẬN NHẦM LẪN (CONFUSION MATRIX)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

res_dir = '/content/roadguard_phase2/yolo11n_v2_unified'
plot_files = ['confusion_matrix.png', 'results.png', 'val_batch0_pred.jpg']

for pf in plot_files:
    img_p = os.path.join(res_dir, pf)
    if os.path.exists(img_p):
        plt.figure(figsize=(10, 6))
        img = mpimg.imread(img_p)
        plt.imshow(img)
        plt.title(pf)
        plt.axis('off')
        plt.show()

### 💾 BƯỚC 7: TẢI MODEL XUẤT XƯỞNG `roadguard_v2_pothole_crack.pt` VỀ MÁY TÍNH

In [ ]:
from google.colab import files

out_filename = 'roadguard_v2_pothole_crack.pt'
shutil.copy2(best_model_path, out_filename)
print(f'✓ Đã chuẩn bị file: {out_filename}')

# Tải tự động về trình duyệt
files.download(out_filename)
print('👉 Hãy lưu file này vào thư mục: weights/trained/roadguard_v2_pothole_crack.pt trong project của bạn!')